# $k\cdot p$ bulk bandstructure of Ge$_{1-x}$Sn$_{x}$

## 1. Setup general settings

### 1.1 Import modules

In [1]:
%reload_ext autoreload
%autoreload 2

In [212]:
from pathlib import Path
import os
import numpy as np
from collections import defaultdict
import h5py
import matplotlib.pyplot as plt
from kpy30band import k_dot_p, read_write_data, plottings

### 1.2 Matplotlib settings

In [149]:
params = {'figure.figsize': (8, 6), 'legend.fontsize': 18, 'axes.labelsize': 24, 'axes.titlesize': 24,
          'xtick.labelsize':24, 'xtick.major.width':2, 'xtick.major.size':5, 'ytick.labelsize': 24,
          'ytick.major.width':2, 'ytick.major.size':5, 'xtick.minor.width':2, 'xtick.minor.size':3,
          'ytick.minor.width':2, 'ytick.minor.size':3, 'errorbar.capsize':2, 'lines.markersize':12,
          'lines.linewidth':2, 'lines.linestyle':'-'}
plt.rcParams.update(params)
plt.rc('font', size=24)
plt_lw=2 # line width for line plots

### 1.3 Set project id (easy to track)

In [4]:
my_project_id = 'kp_simulations' 
sample_id = 'GeSn' # Ge1-xSnx

### 1.4 Save data file path settings

In [11]:
save_f_dir = os.path.join(Path.home(),f'MyFolder/Projects/{my_project_id}/{sample_id}')
save_dataf_dir = f'{save_f_dir}/Data/'
save_figsf_dir = f'{save_f_dir}/Figs/'

os.makedirs(save_dataf_dir, exist_ok=True)
os.makedirs(save_figsf_dir, exist_ok=True)

In [91]:
save_figure = False
FigFormat = 'png'
FigDpi = 300

## 2. Set up simulation system information

### 2.1 General information about the system

In [316]:
# Lattice parameters 
a_Ge = 5.658 # A
a_Sn = 6.489 # A
# Which k-path to plot
kpath_list = ['L-G', 'G-X']
# Number of k-points in each section
nkpoints=41
# Whether to calculate both eigen values or eigenvectors of k.p Hamiltonian
calc_eigenval_only=True
# At which compositions to calculate
x = 0.20

### 2.2 Set $k\cdot p$ parameters

In [317]:
a0=(1.0-x)*a_Ge+x*a_Sn 
E_Gamma = {'EG_2u':17.0426-5.5226*x, 'EG_25u':13.4041-4.8581*x, 'EG_12':8.5786-0.9856*x, 
           'EG_1u':8.2064-2.7334*x, 'EG_1l':-12.2519+1.4249*x, 'EG_15':2.990-0.796*x, 
           'EG_2l':0.8140-3.4667*x+2.2767*x*x, 'EG_25l':0.00}
E_Delta = {'ED_25u':0.0793-0.0333*x, 'ED_15':0.2520+0.193*x, 'ED_25l':0.2247+5.3808*x-4.9535*x*x, 
           'ED_25ul':0.22+0.336*x}
Off_diag = {'P1':0.8421+0.3350*x-0.3346*x*x, 'P2':0.1781-2.1954*x+1.9328*x*x, 
            'P3':-0.0734+1.3200*x-1.2452*x*x, 'P4':1.0543+0.2174*x-0.4220*x*x, 
            'Q1':0.8114-0.1332*x-0.0055*x*x, 'Q2':-0.5334-1.4875*x+2.4647*x*x, 
            'R1':0.3757+0.0340*x+0.0089*x*x, 'R2':0.6820-1.1743*x, 
            'T1':0.7994-0.0565*x-0.0441*x*x, 'T2':-0.0384+0.3675*x} 

### 2.3 Run $k\cdot p$ calculations

#### 2.3.1 Initialize $k\cdot p$ Hamiltonian

In [318]:
kp = k_dot_p(save_file_dir=save_dataf_dir)

#### 2.3.2 Calculate $k\cdot p$ energies at k-points

In [319]:
ek_data = kp.kp_30x30(a0, E_Gamma, E_Delta, Off_diag, kpath_list=kpath_list, 
                      nkpoints=nkpoints, return_eigen_val_vec_both=not calc_eigenval_only,
                      save_data=True, save_file='ek_data.h5')

## 3. Post processing

### 3.1 Load saved data

In [320]:
rw_data_cls = read_write_data(save_file_dir=save_dataf_dir) 

In [321]:
read_data = rw_data_cls.read_data_from_file('ek_data.h5', read_this_k_paths=None) # Default: read all bands

In [322]:
xticks_pos, xticks_label, kpts, bands_e = [], [], [], []
for k_paths in ['L-G', 'G-X']:
    r_data = read_data[k_paths]
    k_ps = r_data[:, 0:3] # First 3 numbers are k-points
    XX = np.sqrt(np.sum(k_ps*k_ps, axis=1))
    if k_paths=='L-G': 
        XX *= -1
        G_line = len(k_ps)
    kpts.append(XX)
    bands_e.append(r_data[:, 3:].T) # 3 is for kx, ky, kz in the begining
    xticks_pos += [XX[0], XX[-1]]
    xticks_label += k_paths.split('-')    
kpts = np.concatenate(kpts, axis=0)
bands_e = np.concatenate(bands_e, axis=1)
special_kpts = {xticks_label[ii]: xticks_pos[ii] for ii in range(len(xticks_label))}

In [323]:
band_ids = {'CB_id': 9, 'VBM_id': 8, 'VB2_id': 5, 'VB3_id': 4} # 

In [324]:
# Python indexting starts with 0
CB_energy = bands_e[band_ids['CB_id'] -1] 
VB_energy = bands_e[band_ids['VBM_id']-1]
G_energy = bands_e[:, G_line]
print(f'Bandgap     = {min(CB_energy) - max(VB_energy) : 0.3f} eV') 
print(f'Bandgap (G) = {G_energy[band_ids['CB_id']-1] - G_energy[band_ids['VBM_id']-1]  : 0.3f} eV') 

Bandgap     =  0.212 eV
Bandgap (G) =  0.212 eV


## 4. Plottings

In [325]:
plt_kp = plottings(save_figure_dir=save_figsf_dir, log_info='1')

### 3.1 Plot band structure

In [326]:
plot_bands = None #None #[1,2,3,4,5,6]
save_fig_file = f'Ge{1-x:0.2f}Sn{x:0.2f}_LGX.{FigFormat}' # file name of the figure to save
set_energy_axis_limit = (-5, 5) #ymin, ymax

In [328]:
fig, ax = plt_kp.plot(kpts, bands_e, ymin=-5, ymax=5, special_kpts=special_kpts, color=None, 
                      savefig=save_figure, save_file_name=save_fig_file, dpi=FigDpi)

* Plot saved: /home/Docs5/badal.mondal/linuxhome/MyFolder/Projects/kp_simulations/GeSn/Figs//Ge0.80Sn0.20_LGX.png
